# Fraud Detection Transformer Model - Experimentation Notebook

This notebook demonstrates how to use the fraud detection transformer model for experimentation and analysis.

## 1. Setup and Imports

In [ ]:
import sys
import os

# Add parent directory to path
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from config.config import MODEL_CONFIG, TRAINING_CONFIG
from src.models.transformer_model import FraudDetectionTransformer
from src.utils.data_preprocessing import FraudDataPreprocessor

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")

## 2. Generate and Explore Data

In [ ]:
# Generate synthetic data
preprocessor = FraudDataPreprocessor()
data = preprocessor.generate_synthetic_data(n_samples=5000, fraud_ratio=0.02)

print(f"Dataset shape: {data.shape}")
print(f"\nFraud ratio: {data['is_fraud'].mean():.2%}")
print(f"\nFirst few rows:")
data.head()

In [ ]:
# Visualize feature distributions
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
features_to_plot = ['amount', 'distance_from_home', 'distance_from_last_transaction', 
                    'ratio_to_median_purchase_price', 'repeat_retailer', 'used_chip']

for idx, feature in enumerate(features_to_plot):
    ax = axes[idx // 3, idx % 3]
    
    normal = data[data['is_fraud'] == 0][feature]
    fraud = data[data['is_fraud'] == 1][feature]
    
    ax.hist(normal, alpha=0.5, label='Normal', bins=30)
    ax.hist(fraud, alpha=0.5, label='Fraud', bins=30)
    ax.set_xlabel(feature)
    ax.set_ylabel('Frequency')
    ax.legend()
    ax.set_title(f'Distribution of {feature}')

plt.tight_layout()
plt.show()

## 3. Build and Examine Model Architecture

In [ ]:
# Build model
fraud_model = FraudDetectionTransformer(MODEL_CONFIG)
model = fraud_model.build_model()

# Display model summary
model.summary()

In [ ]:
# Visualize model architecture
tf.keras.utils.plot_model(
    model,
    to_file='model_architecture.png',
    show_shapes=True,
    show_layer_names=True,
    rankdir='TB',
    expand_nested=True
)

## 4. Prepare Data for Training

In [ ]:
# Prepare train/test split
X_train, X_test, y_train, y_test = preprocessor.prepare_train_test_data(test_size=0.2)

print(f"Training set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")
print(f"Training labels shape: {y_train.shape}")
print(f"Test labels shape: {y_test.shape}")

## 5. Train Model (Quick Demo)

In [ ]:
# Compile model
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy', 
             tf.keras.metrics.Precision(name='precision'),
             tf.keras.metrics.Recall(name='recall'),
             tf.keras.metrics.AUC(name='auc')]
)

# Train for a few epochs (for demo purposes)
history = model.fit(
    X_train, y_train,
    batch_size=32,
    epochs=5,
    validation_data=(X_test, y_test),
    verbose=1
)

## 6. Visualize Training Results

In [ ]:
# Plot training history
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Loss
axes[0, 0].plot(history.history['loss'], label='Training Loss')
axes[0, 0].plot(history.history['val_loss'], label='Validation Loss')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].legend()
axes[0, 0].set_title('Loss over Epochs')

# Accuracy
axes[0, 1].plot(history.history['accuracy'], label='Training Accuracy')
axes[0, 1].plot(history.history['val_accuracy'], label='Validation Accuracy')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Accuracy')
axes[0, 1].legend()
axes[0, 1].set_title('Accuracy over Epochs')

# Precision
axes[1, 0].plot(history.history['precision'], label='Training Precision')
axes[1, 0].plot(history.history['val_precision'], label='Validation Precision')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Precision')
axes[1, 0].legend()
axes[1, 0].set_title('Precision over Epochs')

# Recall
axes[1, 1].plot(history.history['recall'], label='Training Recall')
axes[1, 1].plot(history.history['val_recall'], label='Validation Recall')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Recall')
axes[1, 1].legend()
axes[1, 1].set_title('Recall over Epochs')

plt.tight_layout()
plt.show()

## 7. Evaluate Model Performance

In [ ]:
# Make predictions
y_pred_proba = model.predict(X_test)
y_pred = (y_pred_proba >= 0.5).astype(int)

# Calculate metrics
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc

print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Normal', 'Fraud']))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Normal', 'Fraud'],
            yticklabels=['Normal', 'Fraud'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
# ROC curve
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Classifier')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc='lower right')
plt.grid(alpha=0.3)
plt.show()

## 8. Analyze Predictions

In [ ]:
# Create results dataframe
results_df = pd.DataFrame({
    'actual': y_test,
    'predicted': y_pred.flatten(),
    'probability': y_pred_proba.flatten()
})

# Show some predictions
print("Sample predictions:")
print(results_df.head(20))

# Analyze false positives and false negatives
false_positives = results_df[(results_df['actual'] == 0) & (results_df['predicted'] == 1)]
false_negatives = results_df[(results_df['actual'] == 1) & (results_df['predicted'] == 0)]

print(f"\nFalse Positives: {len(false_positives)}")
print(f"False Negatives: {len(false_negatives)}")

In [ ]:
# Probability distribution by class
plt.figure(figsize=(10, 6))
plt.hist(results_df[results_df['actual'] == 0]['probability'], 
         bins=50, alpha=0.5, label='Normal Transactions', color='blue')
plt.hist(results_df[results_df['actual'] == 1]['probability'], 
         bins=50, alpha=0.5, label='Fraudulent Transactions', color='red')
plt.xlabel('Fraud Probability')
plt.ylabel('Frequency')
plt.title('Distribution of Fraud Probabilities by Actual Class')
plt.legend()
plt.grid(alpha=0.3)
plt.show()